<a href="https://colab.research.google.com/github/Lu-wih/IT-401-GRP-4_FINAL-PROJECT/blob/main/IT_401_G4_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os

In [4]:
from google.colab import drive
drive.mount('/content/drive')

#Import Datasets
df_2024 = pd.read_csv(f"/content/drive/MyDrive/IT 401 - GRP 4_FINAL PROJECT/02_Data /Raw/Datasets (Regression): Annual Traffic/CSV/IT 401 Dataset - AADT - 2024.csv")
df_2025 = pd.read_csv(f"/content/drive/MyDrive/IT 401 - GRP 4_FINAL PROJECT/02_Data /Raw/Datasets (Regression): Annual Traffic/CSV/IT 401 Dataset - AADT - 2025.csv")
df_tomtom = pd.read_csv(f"/content/drive/MyDrive/IT 401 - GRP 4_FINAL PROJECT/02_Data /Raw/Datasets (Regression): Annual Traffic/CSV/Tomtom Traffic data 60 Countries 500 Cities.csv")
gpkg_path = f"/content/drive/MyDrive/IT 401 - GRP 4_FINAL PROJECT/02_Data /Raw/Datasets (Regression): Annual Traffic/CSV/planet_120.775,14.416_121.152,14.717.gpkg"

Mounted at /content/drive


In [5]:
#ADT FILES

In [6]:
#Display the initial datasets
# Check total rows and columns extracted
print("Shape:", df_2024.shape)
print("Shape:", df_2025.shape,"\n")
# Check data types to ensure traffic metrics are numeric
print("Info",df_2024.info(),"\n")
print("Info",df_2025.info(),"\n")
# Display top 5 and bottom 5 rows together
display(df_2024.head())
display(df_2025.head())

Shape: (20, 11)
Shape: (20, 11) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   radial_and_circumferencial_roads  20 non-null     object
 1   car_volume                        20 non-null     int64 
 2   puj_volume                        20 non-null     int64 
 3   uv_volume                         20 non-null     int64 
 4   taxi_volume                       20 non-null     int64 
 5   pub_volume                        20 non-null     int64 
 6   truck_volume                      20 non-null     int64 
 7   trailer_volume                    20 non-null     int64 
 8   mc_volume                         20 non-null     int64 
 9   tricycle_volume                   20 non-null     int64 
 10  total_vehicle                     20 non-null     int64 
dtypes: int64(10), object(1)
memory usage: 1.8+ KB
Info N

,radial_and_circumferencial_roads,car_volume,puj_volume,uv_volume,taxi_volume,pub_volume,truck_volume,trailer_volume,mc_volume,tricycle_volume,total_vehicle
0,c1_recto,30708,9815,823,2884,461,743,94,59551,4650,109729
1,c2_mendoza,54571,35,130,2303,244,7231,847,79105,1745,146211
2,c2_pres_quirino_ave,74152,763,122,3172,403,10020,1521,102419,2566,195138
3,c3_araneta_ave,79018,6,12,1918,614,2718,331,51378,856,136851
4,c4_edsa,210980,1006,2299,10852,3019,2655,50,189078,13,419952


,radial_and_circumferencial_roads,car_volume,puj_volume,uv_volume,taxi_volume,pub_volume,truck_volume,trailer_volume,mc_volume,tricycle_volume,total_vehicle
0,c1_recto,27221,8678,10,1786,63,666,0,67915,9814,116153
1,c2_mendoza,37704,23,5,1694,268,2813,560,53763,2145,98975
2,c2_pres_quirino_ave,79264,447,175,2659,180,8938,1416,104362,2694,200135
3,c3_araneta_ave,77554,3,2,1324,696,2567,409,53046,1368,136969
4,c4_edsa,214080,743,1891,7303,2946,2494,32,192233,6,421728


In [7]:
#Check for missing values
missing = df_2024.isna().sum()
missing[missing > 0]

missing = df_2025.isna().sum()
missing[missing > 0]

,0


In [8]:
# Define target roads and columns to select
target_roads=[
    "c1_recto", "c2_mendonza", "c2_pres_quirino_ave", "r1_roxas_blvd",
    "r2_taft_ave", "r6_magsaysay_blvd", "r9_rizal_ave","r10_del_pan",
0]

selected_columns=[
    "radial_and_circumferencial_roads", "car_volume", "puj_volume",
    "uv_volume", "taxi_volume", "pub_volume", "truck_volume",
    "trailer_volume", "mc_volume", "tricycle_volume", "total_vehicle"
]

In [9]:
# Filtering using defined target roads and columns (ADT2024)
df_ADT2024_filtered = df_2024.loc[
    df_2024["radial_and_circumferencial_roads"].isin(target_roads),selected_columns,].copy()

In [10]:
#Create a clean, lowercase join key without extra spaces (ADT2024)
df_ADT2024_filtered["road_key"] = (
    df_ADT2024_filtered["radial_and_circumferencial_roads"].str.strip().str.lower())

df_ADT2024_filtered.head(30)

,radial_and_circumferencial_roads,car_volume,puj_volume,uv_volume,taxi_volume,pub_volume,truck_volume,trailer_volume,mc_volume,tricycle_volume,total_vehicle,road_key
0,c1_recto,30708,9815,823,2884,461,743,94,59551,4650,109729,c1_recto
2,c2_pres_quirino_ave,74152,763,122,3172,403,10020,1521,102419,2566,195138,c2_pres_quirino_ave
6,r1_roxas_blvd,82145,7860,3683,9689,990,2960,2091,92927,475,202820,r1_roxas_blvd
7,r2_taft_ave,39415,8549,6764,4929,3058,943,5,51620,3836,119119,r2_taft_ave
11,r6_magsaysay_blvd,50465,7062,647,2722,704,4027,1021,91293,2021,159962,r6_magsaysay_blvd
16,r9_rizal_ave,25190,7971,14,1644,390,2069,17,49605,1575,88475,r9_rizal_ave
17,r10_del_pan,34103,692,53,1614,97,5829,6685,63352,3900,116325,r10_del_pan


In [11]:
# Filtering using defined target roads and columns (ADT2024)
df_ADT2025_filtered = df_2025.loc[
    df_2025["radial_and_circumferencial_roads"].isin(target_roads),selected_columns,].copy()

In [12]:
# Create a clean, lowercase join key without extra spaces (ADT2025)
df_ADT2025_filtered["road_key"] = (
    df_ADT2025_filtered["radial_and_circumferencial_roads"].str.strip().str.lower())

df_ADT2025_filtered.head(30)

,radial_and_circumferencial_roads,car_volume,puj_volume,uv_volume,taxi_volume,pub_volume,truck_volume,trailer_volume,mc_volume,tricycle_volume,total_vehicle,road_key
0,c1_recto,27221,8678,10,1786,63,666,0,67915,9814,116153,c1_recto
2,c2_pres_quirino_ave,79264,447,175,2659,180,8938,1416,104362,2694,200135,c2_pres_quirino_ave
6,r1_roxas_blvd,90773,6,2451,4310,1047,1675,4539,98196,271,203268,r1_roxas_blvd
7,r2_taft_ave,54076,11302,6112,3950,3083,1731,6,67068,7686,155014,r2_taft_ave
11,r6_magsaysay_blvd,52387,8017,918,2678,754,4946,1389,88943,1769,161801,r6_magsaysay_blvd
16,r9_rizal_ave,30550,9138,27,1709,452,2582,9,71347,2108,117922,r9_rizal_ave
17,r10_del_pan,36134,556,69,1050,64,6645,9708,59792,2993,117011,r10_del_pan


In [13]:
#CHECKING (AFTER)
# Check total rows and columns extracted
print("Shape:", df_ADT2024_filtered.shape)
print("Shape:", df_ADT2025_filtered.shape)
# Check data types to ensure traffic metrics are numeric
print("\nData Types:\n", df_ADT2024_filtered.dtypes)
print("\nData Types:\n", df_ADT2025_filtered.dtypes)
# Display top 5 and bottom 5 rows together
display(df_ADT2024_filtered.head())
display(df_ADT2024_filtered.tail())
display(df_ADT2025_filtered.head())
display(df_ADT2025_filtered.tail())

Shape: (7, 12)
Shape: (7, 12)

Data Types:
 radial_and_circumferencial_roads    object
car_volume                           int64
puj_volume                           int64
uv_volume                            int64
taxi_volume                          int64
pub_volume                           int64
truck_volume                         int64
trailer_volume                       int64
mc_volume                            int64
tricycle_volume                      int64
total_vehicle                        int64
road_key                            object
dtype: object

Data Types:
 radial_and_circumferencial_roads    object
car_volume                           int64
puj_volume                           int64
uv_volume                            int64
taxi_volume                          int64
pub_volume                           int64
truck_volume                         int64
trailer_volume                       int64
mc_volume                            int64
tricycle_volume          

,radial_and_circumferencial_roads,car_volume,puj_volume,uv_volume,taxi_volume,pub_volume,truck_volume,trailer_volume,mc_volume,tricycle_volume,total_vehicle,road_key
0,c1_recto,30708,9815,823,2884,461,743,94,59551,4650,109729,c1_recto
2,c2_pres_quirino_ave,74152,763,122,3172,403,10020,1521,102419,2566,195138,c2_pres_quirino_ave
6,r1_roxas_blvd,82145,7860,3683,9689,990,2960,2091,92927,475,202820,r1_roxas_blvd
7,r2_taft_ave,39415,8549,6764,4929,3058,943,5,51620,3836,119119,r2_taft_ave
11,r6_magsaysay_blvd,50465,7062,647,2722,704,4027,1021,91293,2021,159962,r6_magsaysay_blvd


,radial_and_circumferencial_roads,car_volume,puj_volume,uv_volume,taxi_volume,pub_volume,truck_volume,trailer_volume,mc_volume,tricycle_volume,total_vehicle,road_key
6,r1_roxas_blvd,82145,7860,3683,9689,990,2960,2091,92927,475,202820,r1_roxas_blvd
7,r2_taft_ave,39415,8549,6764,4929,3058,943,5,51620,3836,119119,r2_taft_ave
11,r6_magsaysay_blvd,50465,7062,647,2722,704,4027,1021,91293,2021,159962,r6_magsaysay_blvd
16,r9_rizal_ave,25190,7971,14,1644,390,2069,17,49605,1575,88475,r9_rizal_ave
17,r10_del_pan,34103,692,53,1614,97,5829,6685,63352,3900,116325,r10_del_pan


,radial_and_circumferencial_roads,car_volume,puj_volume,uv_volume,taxi_volume,pub_volume,truck_volume,trailer_volume,mc_volume,tricycle_volume,total_vehicle,road_key
0,c1_recto,27221,8678,10,1786,63,666,0,67915,9814,116153,c1_recto
2,c2_pres_quirino_ave,79264,447,175,2659,180,8938,1416,104362,2694,200135,c2_pres_quirino_ave
6,r1_roxas_blvd,90773,6,2451,4310,1047,1675,4539,98196,271,203268,r1_roxas_blvd
7,r2_taft_ave,54076,11302,6112,3950,3083,1731,6,67068,7686,155014,r2_taft_ave
11,r6_magsaysay_blvd,52387,8017,918,2678,754,4946,1389,88943,1769,161801,r6_magsaysay_blvd


,radial_and_circumferencial_roads,car_volume,puj_volume,uv_volume,taxi_volume,pub_volume,truck_volume,trailer_volume,mc_volume,tricycle_volume,total_vehicle,road_key
6,r1_roxas_blvd,90773,6,2451,4310,1047,1675,4539,98196,271,203268,r1_roxas_blvd
7,r2_taft_ave,54076,11302,6112,3950,3083,1731,6,67068,7686,155014,r2_taft_ave
11,r6_magsaysay_blvd,52387,8017,918,2678,754,4946,1389,88943,1769,161801,r6_magsaysay_blvd
16,r9_rizal_ave,30550,9138,27,1709,452,2582,9,71347,2108,117922,r9_rizal_ave
17,r10_del_pan,36134,556,69,1050,64,6645,9708,59792,2993,117011,r10_del_pan


In [14]:
#Create dedicated output directory if it doesn't exist
output_dir = f"/content/drive/MyDrive/IT 401 - GRP 4_FINAL PROJECT/02_Data /Processed"
os.makedirs(output_dir, exist_ok=True)

#Export cleaned AADT CSVs
df_ADT2024_filtered.to_csv(f"{output_dir}/AADT_2024_cleaned.csv",index=False)
df_ADT2025_filtered.to_csv(f"{output_dir}/AADT_2025_cleaned.csv",index=False)
print("Cleaned datasets saved successfully!")

Cleaned datasets saved successfully!


In [15]:
#TomTom Files

In [16]:
# Filtering using defined target roads and columns (TomTom)
tomtom_cols=["City","UpdateTimeUTC","JamsDelay","TrafficIndexLive","JamsLengthInKms","TravelTimeLivePer10KmsMins","TravelTimeHistoricPer10KmsMins","MinsDelay",]
df_tomtom_filtered = df_tomtom.loc[df_tomtom["City"].str.lower() == "manila", tomtom_cols].copy()

print(list(df_tomtom.columns))

['Country', 'City', 'UpdateTimeUTC', 'JamsDelay', 'TrafficIndexLive', 'JamsLengthInKms', 'JamsCount', 'TrafficIndexWeekAgo', 'UpdateTimeUTCWeekAgo', 'TravelTimeLivePer10KmsMins', 'TravelTimeHistoricPer10KmsMins', 'MinsDelay']


In [17]:
#CHECKING (Before/Double Checking)
# Check total rows and columns extracted
print("Shape:", df_tomtom_filtered.shape)
# Check data types to ensure traffic metrics are numeric
print("\nData Types:\n", df_tomtom_filtered.dtypes)
# Display top 5 and bottom 5 rows together
display(df_tomtom_filtered.head())
display(df_tomtom_filtered.tail())

Shape: (2668, 8)

Data Types:
 City                               object
UpdateTimeUTC                      object
JamsDelay                         float64
TrafficIndexLive                  float64
JamsLengthInKms                   float64
TravelTimeLivePer10KmsMins        float64
TravelTimeHistoricPer10KmsMins    float64
MinsDelay                         float64
dtype: object


,City,UpdateTimeUTC,JamsDelay,TrafficIndexLive,JamsLengthInKms,TravelTimeLivePer10KmsMins,TravelTimeHistoricPer10KmsMins,MinsDelay
654621,manila,2025-01-13 04:01:00.000,1511.0,62.0,216.4,26.680556,25.775926,0.904630
654622,manila,2025-01-13 03:01:00.000,1502.5,65.0,232.9,26.953219,26.175298,0.777921
654623,manila,2025-01-13 02:01:00.000,1263.6,64.0,199.4,26.774737,26.278764,0.495973
654624,manila,2025-01-13 01:01:05.797,1221.1,73.0,188.4,28.622719,27.892405,0.730314
654625,manila,2025-01-13 00:01:04.910,1679.0,85.0,254.5,30.545980,29.220405,1.325575


,City,UpdateTimeUTC,JamsDelay,TrafficIndexLive,JamsLengthInKms,TravelTimeLivePer10KmsMins,TravelTimeHistoricPer10KmsMins,MinsDelay
657284,manila,2024-10-01 11:01:14.678,3446.8,122.0,479.4,37.249580,33.853464,3.396116
657285,manila,2024-10-01 10:31:13.331,4614.0,127.0,631.7,38.174732,34.752339,3.422394
657286,manila,2024-10-01 09:45:53.238,4545.8,118.0,643.6,36.711922,34.280340,2.431582
657287,manila,2024-10-01 08:46:00.000,3491.3,103.0,514.2,34.164499,31.727207,2.437292
657288,manila,2024-10-01 07:46:00.001,3147.7,91.0,443.8,31.889696,29.824424,2.065272


In [18]:
#Convert UpdateTimeUTC to Philippine Time (PHT)
if "UpdateTimeUTC" in df_tomtom_filtered.columns:
    df_tomtom_filtered["UpdateTimePHT"] = (
        pd.to_datetime(df_tomtom_filtered["UpdateTimeUTC"])
        .dt.tz_localize("UTC")
        .dt.tz_convert("Asia/Manila")
        .dt.tz_localize(None)
    )
    df_tomtom_filtered = df_tomtom_filtered.drop(columns=["UpdateTimeUTC"])

#Convert JamsDelay to minutes
if "JamsDelay" in df_tomtom_filtered.columns:
    df_tomtom_filtered["JamsDelayMins"] = df_tomtom_filtered["JamsDelay"] / 60
    df_tomtom_filtered = df_tomtom_filtered.drop(columns=["JamsDelay"])
elif "JamsDelaySecs" in df_tomtom_filtered.columns:
    df_tomtom_filtered["JamsDelayMins"] = (
        df_tomtom_filtered["JamsDelaySecs"] / 60
    )
    df_tomtom_filtered = df_tomtom_filtered.drop(columns=["JamsDelaySecs"])

#Move JamsDelayMins to 3rd column position (index 2)
if "JamsDelayMins" in df_tomtom_filtered.columns:
    cols = df_tomtom_filtered.columns.tolist()
    cols.insert(2, cols.pop(cols.index("JamsDelayMins")))
    df_tomtom_filtered = df_tomtom_filtered[cols]

#Move UpdateTimePHT to the 2nd column position (index 1)
if "UpdateTimePHT" in df_tomtom_filtered.columns:
    cols = df_tomtom_filtered.columns.tolist()
    cols.insert(1, cols.pop(cols.index("UpdateTimePHT")))
    df_tomtom_filtered = df_tomtom_filtered[cols]

# Rename JamsDelayMins to CausedDelayMins
if "MinsDelay" in df_tomtom_filtered.columns:
    df_tomtom_filtered = df_tomtom_filtered.rename(
        columns={"MinsDelay": "CausedDelayMins"}
    )

df_tomtom_filtered.head()

,City,UpdateTimePHT,TrafficIndexLive,JamsDelayMins,JamsLengthInKms,TravelTimeLivePer10KmsMins,TravelTimeHistoricPer10KmsMins,CausedDelayMins
654621,manila,2025-01-13 12:01:00.000,62.0,25.183333,216.4,26.680556,25.775926,0.904630
654622,manila,2025-01-13 11:01:00.000,65.0,25.041667,232.9,26.953219,26.175298,0.777921
654623,manila,2025-01-13 10:01:00.000,64.0,21.060000,199.4,26.774737,26.278764,0.495973
654624,manila,2025-01-13 09:01:05.797,73.0,20.351667,188.4,28.622719,27.892405,0.730314
654625,manila,2025-01-13 08:01:04.910,85.0,27.983333,254.5,30.545980,29.220405,1.325575


In [19]:
#Create dedicated output directory if it doesn't exist
output_dir = f"/content/drive/MyDrive/IT 401 - GRP 4_FINAL PROJECT/02_Data /Processed"
os.makedirs(output_dir, exist_ok=True)

#Export cleaned AADT CSVs
df_tomtom_filtered.to_csv(f"{output_dir}/TomtomManila_cleaned.csv",index=False)
print("Cleaned datasets saved successfully!")

Cleaned datasets saved successfully!


**Regression**

In [20]:


# 2. Compute custom percentage Total_Score (USE THIS FOR )
#df_StuRec['Total_Score'] = (
 #   df_StuRec['Attendance (%)'] * 0.20 +
  #  study_hours_scaled * 0.10 +
   # df_StuRec['Assignments_Avg'] * 0.40 +
    #df_StuRec['Quizzes_Avg'] * 0.30

Use this for vehicle composition (Base on PCU)